In [5]:
import math

In [6]:
def max_bid(sell_price, fee_rate=0.15, shipping=6.0):
    fraction = 0.75 - 0.273 * math.exp(-0.02 * sell_price)
    bid = sell_price * fraction - shipping
    return round(bid, 2) if bid > 0 else None

In [14]:
max_bid(49.99)

26.47

In [8]:
for i in range(5, 50, 5):
    print(max_bid(i))

None
None
2.22
5.34
8.61
12.01
15.51
19.09
22.76


In [9]:
def max_discogs_price(sell_price, fee_rate=0.15):
    """Same margin curve as max_bid, but for Discogs listings where
    shipping is already included in the price (most good sellers ship
    free) -- so there is no separate shipping cost to subtract."""
    fraction = 0.75 - 0.273 * math.exp(-0.02 * sell_price)
    price = sell_price * fraction
    return round(price, 2) if price > 0 else None

In [10]:
def max_discogs_price(sell_price):
    """Same margin curve as max_bid, but for Discogs listings where
    shipping is already included in the price (most good sellers ship
    free) -- so there is no separate shipping cost to subtract."""
    fraction = 0.75 - 0.273 * math.exp(-0.02 * sell_price)
    price = sell_price * fraction
    return round(price, 2) if price > 0 else None

In [11]:
def discogs_sell_price(buy_price, tol=0.01):
    """Inverse of max_discogs_price: given what you actually paid,
    find the sell price that maintains the same margin curve.
    sell_price appears both linearly and inside the exponential term
    in max_discogs_price, so there is no closed-form inverse --
    solved numerically instead (bisection; the relationship is
    monotonically increasing so this is safe)."""
    if buy_price <= 0:
        return None
    lo, hi = 0.0, max(100.0, buy_price * 5)
    while hi - lo > tol:
        mid = (lo + hi) / 2
        implied_buy = max_discogs_price(mid) or 0
        if implied_buy < buy_price:
            lo = mid
        else:
            hi = mid
    return round(hi, 2)

In [13]:
discogs_sell_price(50)

72.84

In [ ]:
for buy in [5, 10, 15, 20, 30, 50, 100]:
    sell = discogs_sell_price(buy)
    print(f"paid {buy:>5} -> sell {sell:>7} (check: max_discogs_price(sell)={max_discogs_price(sell)})")